# C8-embeddings — Session 3: Nearest Neighbors and Retrieval

*One class session, roughly 85 minutes. Builds on Sessions 1–2 (tokens,
the unit-row stack `W`, the similarity matrix $S = W W^{\mathsf T}$).*

**This session:** the one NumPy tool this unit still lacks —
**`np.argsort`**, taught from scratch; turning a similarity row into a
ranked neighbor list (descending order, **self-exclusion**, top-$k$);
cross-checking our manual ranking against gensim's `most_similar` and
getting an *exact* match; what similarity rankings are quietly biased
by (frequency and hubness — stated facts); and the session's
destination, a complete **retrieval mini-pipeline**: query → tokenize →
filter → embed → rank, end to end on fresh text.
Plus this unit's Exam Connections and Going Deeper.

Try every checkpoint by hand first, then verify in code.
Answers are collected at the end of this notebook.

In [ ]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
from gensim.utils import simple_preprocess
import gensim.downloader

kv = gensim.downloader.load("glove-wiki-gigaword-100")

# Session 2's stack, rebuilt in three lines (the pinned conventions)
WORDS = ["harbor", "boat", "sailor", "tide", "violin", "cello", "pepper", "honey"]
V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
S = W @ W.T
print("S ready:", S.shape)

## 1. The Missing Tool: `np.argsort`

**Motivation.**
"Which words are most similar to `harbor`?" means: sort row `S[0]` —
but keep track of *which entries* ended up where.
Sorting the values loses the identities; we need the sorted **indices**.

**Definition (`np.argsort`).**
`np.argsort(a)` returns the array of indices that would sort `a` in
*ascending* order: position 0 holds the index of `a`'s smallest entry,
position 1 the index of the next smallest, and so on.
`a[np.argsort(a)]` is therefore `a` sorted.
Two idioms complete the toolkit:

- **descending order**: reverse with slicing — `np.argsort(a)[::-1]`
  (largest entry's index first);
- **read a ranking**: `order[0]` is "who came first", `order[:k]` the
  top $k$.

On a checkable toy first.

In [ ]:
a = np.array([3.0, 1.0, 2.0])

order_up = np.argsort(a)
print("ascending indices :", order_up)        # [1 2 0]: a[1]=1 < a[2]=2 < a[0]=3
print("a sorted          :", a[order_up])

order_down = order_up[::-1]
print("descending indices:", order_down)      # [0 2 1]
print("a sorted (desc)   :", a[order_down])

Read the ascending output aloud: "the smallest entry lives at index 1,
then index 2, then index 0."
`argsort` answers *where*, never *what* — the values come from indexing
`a` with the result.
(Ties are broken by position, earlier index first; with real-valued
cosines, exact ties essentially never occur.)

### Checkpoint 1

1. By hand: `s = np.array([0.2, 0.9, 0.4, 1.0])`.
   Write `np.argsort(s)`, `np.argsort(s)[::-1]`, and the *top two*
   indices by value.
2. Without running: what is `a[np.argsort(a)][-1]` for any 1-D array
   `a`, in words?
3. A classmate uses `np.sort(S[0])[::-1][:3]` to report "the three
   nearest words to `harbor`". What do they actually have, and what
   can't they do with it?

## 2. Nearest Neighbors in a Similarity Row

Row $i$ of $S$ holds token $i$'s cosine with every token — itself
included.
Ranking it descending puts the *query itself* first ($S_{ii} = 1$ is
always the row maximum), so every honest neighbor list starts with
**self-exclusion**: drop index $i$ from the order, then take the
top $k$.
The mask idiom `order[order != i]` does it robustly — it removes $i$
*wherever* it sits, rather than assuming it sits first.

In [ ]:
i = WORDS.index("harbor")
order = np.argsort(S[i])[::-1]
print("raw descending order:", [WORDS[j] for j in order])

neighbors = order[order != i]            # self-exclusion, position-safe
top3 = neighbors[:3]
print("top 3 neighbors of 'harbor':",
      [(WORDS[j], round(float(S[i, j]), 4)) for j in top3])

In [ ]:
def top_k(S, i, k):
    """Indices of the k nearest tokens to token i (self excluded)."""
    order = np.argsort(S[i])[::-1]
    return order[order != i][:k]


for w in ["violin", "honey"]:
    j = WORDS.index(w)
    idx = top_k(S, j, 3)
    print(f"{w:7} ->", [(WORDS[t], round(float(S[j, t]), 4)) for t in idx])

`harbor`'s neighbors — `boat` ($0.6023$), `tide`, `sailor` — are its
sea block from Session 2's heatmap, now ranked.
`violin` finds `cello` at $0.925$; `honey` finds `pepper`.
The matrix view and the neighbor view are the same information, read
along a row.

### Checkpoint 2

1. Why is the raw descending order's first entry *always* the query
   itself? Cite the property of $S$ that guarantees it.
2. `order[1:k+1]` also "drops the self" and is one character shorter
   than the mask idiom. Construct the circumstance where it silently
   returns a wrong list. (Hint: what if two entries tie at $1.0$?)
3. By hand: for `s_row = [1.0, 0.3, 0.8, 0.5]` and query index 0,
   apply the full recipe (argsort desc, exclude, top-2) and give the
   resulting indices.

## 3. Cross-Checking Against `most_similar`

`gensim` ships the same computation as a method:
`kv.most_similar(word, topn=k)` returns the $k$ nearest vocabulary
words by cosine, query excluded — our recipe, run over the *whole*
400,000-word vocabulary.

One stated fact makes the comparison cheap: **GloVe's vocabulary is
ordered by corpus frequency** — index 0 is the most frequent word, and
`kv.vectors[:20000]` is exactly the 20,000 most frequent words.
`most_similar` accepts `restrict_vocab=20000` to search only that
slice, so we can rebuild its answer manually: stack those 20,000 rows,
normalize, rank against the query's unit vector, exclude the query's
own index.
If our machinery is right, the two lists must agree word for word —
and they do.

In [ ]:
M = 20000
Vm = np.asarray(kv.vectors[:M], dtype=np.float64)      # boundary cast
Wm = Vm / np.sqrt((Vm * Vm).sum(axis=1, keepdims=True))

q = np.asarray(kv["harbor"], dtype=np.float64)
q = q / np.sqrt((q * q).sum())

sims = Wm @ q                                          # (20000,)
iq = kv.key_to_index["harbor"]
order = np.argsort(sims)[::-1]
order = order[order != iq]
manual = [(kv.index_to_key[j], round(float(sims[j]), 4)) for j in order[:5]]
print("manual  :", manual)

gens = kv.most_similar("harbor", topn=5, restrict_vocab=M)
print("gensim  :", [(w, round(float(s), 4)) for w, s in gens])

gap = max(abs(float(sims[kv.key_to_index[w]]) - float(s)) for w, s in gens)
print("largest similarity gap:", gap)
assert [w for w, _ in manual] == [w for w, _ in gens]
assert gap < 1e-5

Word-for-word agreement — `harbour`, `bay`, `shore`, `shores`,
`pearl` — with similarity gaps around $10^{-7}$.

**Why not exactly zero?**
gensim runs its arithmetic in float32 (the artifact's native dtype);
we run in float64 after the boundary cast.
Same formula, different rounding — agreement to about seven digits is
*correct behavior*, which is why cross-checks against gensim values use
`np.isclose(..., atol=1e-5, rtol=0)` rather than equality, while anchors
computed *within* our own float64 pipeline can be held to `1e-6` and
tighter.

### Checkpoint 3

1. In the manual rebuild, why must the query's *index* be excluded
   from the order even though the query was never appended to `Wm`?
2. `most_similar("harbor", topn=5)` (no `restrict_vocab`) could return
   a slightly different list than the restricted call. Explain how,
   using the frequency-ordering fact.
3. Pick the right tolerance and justify in one sentence: comparing
   your float64 cosine against (a) `kv.similarity`'s float32 value,
   (b) another float64 value computed from the same `W`.

## 4. What Neighbor Lists Are Biased By

Two stated facts to carry into any embedding project — no derivations,
just calibrated skepticism:

**Fact 1 — frequency effects.**
Very frequent words (`the`, `of`, `one`) occur in *every* context, so
their vectors encode little topical meaning; their neighbors are other
high-frequency function words, not topics.
Watch it happen:

In [ ]:
print("neighbors of 'the':",
      [w for w, _ in kv.most_similar("the", topn=5, restrict_vocab=20000)])

`this`, `part`, `one`, `of`, `same` — grammatically ubiquitous,
topically empty.
A retrieval pipeline that does not filter function words will find them
"similar" to everything; real systems drop them (stop-word lists) or
weight them down.

**Fact 2 — hubness.**
In high-dimensional spaces some points end up unusually *central*,
appearing in the top-$k$ lists of very many other points — such points
are called **hubs**.
A word that keeps showing up as everyone's neighbor is not necessarily
meaningful; it may just sit near the center of the point cloud.
The practical rule from both facts: **read a neighbor list as evidence
to inspect, not as an answer to trust** — check the actual similarity
values, not only the ranks.

### Checkpoint 4

1. Your pipeline reports that `one` is a top-3 neighbor of eleven of
   your twelve query words. Which fact(s) explain it, and what is the
   cheapest fix?
2. Two neighbor lists both rank `breakwater` first — with similarity
   $0.81$ in one case and $0.31$ in the other. Why does the stated
   rule ("check values, not only ranks") treat these differently?

## 5. The Retrieval Mini-Pipeline

Everything in this unit, composed into one function.
**Task:** given a query word and a passage, rank the passage's
vocabulary by similarity to the query.
The five stages are exactly the sections you have already built:

1. **tokenize** the passage (`simple_preprocess`, Session 1 §1);
2. **deduplicate, order kept** (`dict.fromkeys`, Session 1 §2);
3. **filter** to in-vocabulary tokens (Session 1 §5);
4. **embed**: stack candidate vectors, float64, unit rows
   (Session 2 §§1–3);
5. **rank**: similarities to the query's unit vector, `argsort`
   descending (this session).

The query is embedded separately and is *not* a row of the candidate
matrix, so no self-exclusion is needed here — one honest difference
from Section 2's within-matrix ranking.

In [ ]:
def retrieve(kv, query, passage, k):
    """Top-k passage words by cosine similarity to the query word."""
    toks = simple_preprocess(passage)                              # 1. tokenize
    cands = [t for t in dict.fromkeys(toks)                        # 2. dedup, ordered
             if t in kv.key_to_index]                              # 3. filter
    V = np.asarray(kv[cands], dtype=np.float64)                    # 4. embed...
    Wp = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
    q = np.asarray(kv[query], dtype=np.float64)
    q = q / np.sqrt((q * q).sum())
    sims = Wp @ q                                                  # 5. rank
    order = np.argsort(sims)[::-1]
    return [(cands[j], float(sims[j])) for j in order[:k]]


PASSAGE = ("Storm warnings kept every boat inside the harbor. A sailor coiled rope on "
           "the pier while rain drummed against the lighthouse windows. By evening the "
           "tide had buried the sandbar, and the ferry waited for calmer water.")

for word, sim in retrieve(kv, "ocean", PASSAGE, 5):
    print(f"  {word:10} {sim:.4f}")

Query `ocean` — a word that never appears in the passage — surfaces
`water`, `harbor`, `boat`, `storm`, `rain`: the passage's maritime core,
ranked by geometry alone.
No keyword matched; the embedding *is* the bridge between the query and
the text.
This five-stage shape — tokenize, dedup, filter, embed, rank — is the
skeleton of every retrieval system you will meet, however large.

### Checkpoint 5

1. Stage order matters. What goes wrong if stage 3 (filter) runs
   *after* stage 4's `kv[cands]` lookup?
2. Why is stage 2 `dict.fromkeys` rather than `set` — which later
   stage would a `set` corrupt, and how would the bug manifest?
3. Modify one line of `retrieve` so it returns the *least* similar
   k words instead. (Two different one-line edits work; give either.)

## 6. Worked Exam-Style Example: Constrained Coding

The exam register, worked in full: exact contract, explicit bans,
zero-points clause.

---

**Problem.**
Write `rank_by_query(kv, words, query)` returning the list `words`
re-ordered by descending cosine similarity to `query` (all inputs
in-vocabulary; `query` not in `words`).

**Banned (zero points): any `np.linalg` function, any Python loop or
comprehension over array elements, `np.einsum`, `np.tensordot`,
`sklearn`, `scipy`, and every gensim similarity helper
(`kv.most_similar`, `kv.similarity`, `kv.n_similarity`,
`kv.distances`).**
Vector lookup `kv[...]` is allowed — it is the load boundary.

---

**Step 1 — plan against the bans.**
The bans leave exactly the course register: boundary-cast lookup,
broadcasting norms with `keepdims`, a matrix–vector product, `argsort`.
That is the whole solution — the bans *are* the outline.

**Step 2 — write it, one pinned convention per line.**

In [ ]:
def rank_by_query(kv, words, query):
    V = np.asarray(kv[words], dtype=np.float64)             # (N,100), float64
    Wq = V / np.sqrt((V * V).sum(axis=1, keepdims=True))    # unit rows
    q = np.asarray(kv[query], dtype=np.float64)
    q = q / np.sqrt((q * q).sum())                          # unit query
    sims = Wq @ q                                           # (N,) cosines
    order = np.argsort(sims)[::-1]                          # descending
    return [words[j] for j in order]                        # list-comp over
                                                            # words: allowed


print(rank_by_query(kv, ["cello", "salmon", "thunder", "cheese"], "flute"))

# smoke-test the values behind the ranking (display only)
Vx = np.asarray(kv[["cello", "salmon", "thunder", "cheese"]], dtype=np.float64)
Wx = Vx / np.sqrt((Vx * Vx).sum(axis=1, keepdims=True))
qx = np.asarray(kv["flute"], dtype=np.float64)
print(np.round(Wx @ (qx / np.sqrt((qx * qx).sum())), 4))

**Step 3 — sanity-check the output.**
`cello` first (a fellow instrument, cosine $0.831$ in the smoke-test
line), then `salmon`, `thunder`, `cheese` trailing far behind (all
under $0.2$) — the ranking agrees with common sense, which is the
fastest smoke test a constrained-coding answer gets.
Note the final list comprehension runs over the Python list `words`,
not over array entries — reading indices *out* of a ranking is list
work, and the ban clause says "over array elements" for exactly this
reason.
When in doubt, the arithmetic (norms, dots, sums) must be pure
broadcasting; the bookkeeping (labels in, labels out) may be Python.

### Checkpoint 6

1. Which single line of `rank_by_query` changes if `query` may appear
   in `words`, and to what?
2. Why does the bans list include `kv.most_similar` here, when
   Section 3 celebrated it? What is each register *for*?

## 7. Common Pitfalls III

**Pitfall 1 — ascending when you meant descending.**
`np.argsort` sorts ascending; forgetting `[::-1]` hands you the
*least* similar words, in the most confident tone.

In [ ]:
i = WORDS.index("harbor")
wrong = np.argsort(S[i])[:3]                       # forgot [::-1]
right = top_k(S, i, 3)
print("forgot [::-1]  :", [WORDS[j] for j in wrong], " <- least similar!")
print("correct        :", [WORDS[j] for j in right])

The broken version returns real words with real similarities — nothing
raises.
The tell: the query's own index sits *last* in a full ascending order,
and the reported "neighbors" have the row's smallest values.
Print the similarity values alongside the words and the bug confesses
instantly.

**Pitfall 2 — forgetting self-exclusion.**
Descending order without the mask reports the query as its own best
neighbor ($S_{ii} = 1$) and silently shifts every real neighbor down
one slot — so a requested top-3 delivers only two useful answers.
Both pitfalls at once is the classic broken ranking: p16 hands you
exactly that function to diagnose.

**Pitfall 3 — ranking with the wrong universe.**
A ranking is only defined relative to its candidate set.
`most_similar` ranks 400,000 words; Section 2 ranked eight; the
pipeline ranked one passage's vocabulary.
Comparing their outputs without aligning universes (as Section 3 did
with `restrict_vocab`) produces "mismatches" that are not bugs — and
occasionally agreement that is pure luck.
State the universe before comparing lists.

**Pitfall 4 — the query inside its own candidate list.**
If the query word happens to appear in the passage, the pipeline of
Section 5 will rank it — at similarity $1.0$, permanently first.
Whether that is a bug depends on the task ("find *other* related
words" vs "find matching words"); the pitfall is not *deciding*.
The fix, when you want it: filter the query from the candidates, or
exclude it after ranking like Section 2.

### Checkpoint 7

1. A ranking function passes every test whose queries are absent from
   the passage, then "fails" the first time a query word occurs in the
   text. Which pitfall, and what are the two defensible fixes?
2. Your manual top-5 disagrees with `most_similar`'s top-5 at
   positions 4–5 only. Which pitfall does Section 3's methodology rule
   out first, and how?
3. Sketch the two-line print that would have exposed Pitfall 1 on
   first run.

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic tables — no real test text here):

- The **NLP/embeddings cluster** (tokenization, dedup semantics,
  cosine-similarity meaning) appears as early sub-parts of the exam's
  big multi-part arc: exactly Session 1's material — token/type
  counting after a standardizing tokenizer, what `set()` loses
  (order and counts), and reading cosine values as relatedness.
  p01, p06, and p15 drill that register.
- The exam's **dominant 90-point arc** builds embeddings → similarity
  → SVD → low-rank: its front half is this unit verbatim — stack
  vectors into a matrix, normalize rows *under explicit API bans*
  (no `np.linalg`, no loops), form the Gram/similarity matrix, rank
  neighbors.
  The arc texture — later parts consuming earlier results — is the
  shape of p13 and p14.
- The **linear-algebra cluster** (10 sub-parts, 60 points) includes
  Gram-matrix reasoning; Session 2 §5's entrywise derivation (graded
  as p11) and the cosine-range proof (p12) are its register, with
  "Reasoning is required" flags.
- **Tooling surface**: the exam names gensim GloVe embeddings
  explicitly, with starter code provided — Session 1 §4's load /
  membership / lookup calls and Section 3's `most_similar`
  cross-check are the fluency being assumed; p05, p13, p18, and p19
  train it.
- **Grading signals**: zero-point clauses for ban violations and
  numeric normal forms (Session 2 §7) — practice states both in the
  exam's own wording.

## Going Deeper

Optional forward pointers along the course map — nothing here is
needed for this unit's practice:

- **`C9-dimensionality-reduction`** — the direct sequel, and the back
  half of the exam arc this unit's front half started.
  It consumes *exactly* this unit's pinned object: the unit-row stack
  `W`, rows-are-tokens, shape $(N, d)$, with $S = W W^{\mathsf T}$.
  There the SVD factors `W`, best low-rank approximations compress
  it, and the block structure you *saw* in Session 2's heatmap
  becomes something you can *count* (how many directions carry the
  sea block? the strings block?).
- **`C5-neural-networks` (already behind you)** — embeddings are how
  text enters the network family that unit built: an embedding matrix
  is the first layer of essentially every modern language model,
  looked up by token index exactly as `kv[...]` did here. The vectors
  there are *trained with* the network rather than loaded fixed — the
  geometry vocabulary (similarity, neighbors, hubs) transfers
  unchanged.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. `np.argsort(s)` = `[0, 2, 1, 3]`; reversed = `[3, 1, 2, 0]`;
   top two indices: `3` (value $1.0$) and `1` (value $0.9$).
2. The largest value of `a` — sorting ascending puts the maximum last.
3. They have the three largest similarity *values* (e.g. $1.0, 0.602,
   0.328$) with no idea which words they belong to — values without
   identities can be reported but not acted on.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $S_{ii} = 1$ (unit diagonal), and every entry of $S$ is a cosine
   $\le 1$ — so the diagonal entry is the row's maximum and sorts
   first.
2. If some other token has cosine exactly $1.0$ with the query (a
   duplicated word in the list, or a genuine tie), argsort may place
   *it* first and the query second; `order[1:k+1]` then drops the
   duplicate and *keeps the query itself* in the list. The mask drops
   the query wherever it sits.
3. Descending order of values $(1.0, 0.8, 0.5, 0.3)$ is indices
   `[0, 2, 3, 1]`; excluding query 0 leaves `[2, 3, 1]`; top-2 =
   `[2, 3]`.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. The query is one of the 20,000 most frequent words, so its own row
   *is* in `Wm` (at index `key_to_index["harbor"]`) — it would rank
   first at similarity 1 like any diagonal hit.
2. The unrestricted call searches 380,000 additional, rarer words;
   any of them with cosine above the restricted list's fifth value
   displaces it. Restriction can only remove candidates, never add.
3. (a) `atol=1e-5, rtol=0` — float32 arithmetic agrees with float64 only to
   ~7 significant digits; (b) `atol=1e-6` or tighter — both numbers
   come from the same float64 pipeline, so only float64 rounding
   separates them.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Fact 1 (frequency): `one` is a top-30 corpus word whose vector is
   topically empty; hubness can amplify it. Cheapest fix: drop a
   stop-word list (or the top few hundred most frequent words) from
   the candidates before ranking.
2. Rank 1 at $0.81$ is strong evidence of relatedness; rank 1 at
   $0.31$ merely means "nothing in this candidate set is close" —
   the rank is the same, the evidence is not. Values calibrate;
   ranks only order.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. `kv[cands]` raises `KeyError` on the first OOV token — the lookup
   *is* the operation the filter exists to protect.
2. A `set` scrambles candidate order, so the rows of the stacked
   matrix no longer correspond to the list you keep — every similarity
   would be attributed to the wrong word on display. (The bug is
   nondeterministic across runs, the worst kind.)
3. Either drop the reversal (`order = np.argsort(sims)`) or negate
   the scores (`sims = -(Wp @ q)`); then `order[:k]` is the bottom-k.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. The last line, to mask before reading:
   `return [words[j] for j in order if words[j] != query]` — or
   equivalently exclude by index after locating the query in `words`.
2. `most_similar` is the *verification* register (gensim-usage
   problems require it); the manual register is the *construction*
   being graded. Using the library inside the construction answers a
   different question from the one asked — hence zero points.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Pitfall 4. Fix A: remove the query from the candidates before
   embedding; fix B: rank everything, then drop the query from the
   output (Section 2's mask). Either is fine once *chosen on
   purpose*.
2. Pitfall 3 (universe mismatch): Section 3 aligns the universes with
   `restrict_vocab` before comparing, so any remaining disagreement
   is a real bug, not a candidate-set artifact.
3. `print([WORDS[j] for j in result])` alongside
   `print(np.round(S[i][result], 3))` — least-similar words carry
   visibly tiny values next to their names.

</details>